# Research Tasks 1–4: Concept-Aware Training Extensions

This notebook runs the four research tasks agreed on with Chen (May 2026).

**Prerequisites:** The three baseline checkpoints (standard CLM, synonym NCP, hypernym NCP) must already exist in your Drive output folder from the previous training runs.

| Task | Description | Status |
|------|-------------|--------|
| **Task 1** | Rerun NTP eval on vanilla val data (all checkpoints) | ⬜ Not run |
| **Task 2** | Differentiable NCP training (syn + hyp) | ⬜ Not run |
| **Task 3** | Build contrastive dataset with hard negatives | ⬜ Not run |
| **Task 4** | Contrastive training + final comparison | ⬜ Not run |

**Runtime:** Use the A100 on Irad's server. Do not use free T4 for Tasks 2 and 4 (training will be slow).

---
## Setup

Run all setup cells top-to-bottom before starting any task.

In [ ]:
# S1: Check GPU
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(result.stdout.strip())

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# S2: Clone repo and install dependencies
REPO_URL = "https://github.com/SharvaGogawale1/concept-aware-training.git"

import os
if not os.path.exists('/content/concept_aware_training'):
    !git clone {REPO_URL} /content/concept_aware_training
else:
    print("Repo already cloned — pulling latest")
    !cd /content/concept_aware_training && git pull

# Install dependencies (skip conda lock file, pin transformers)
!pip install -q \
    transformers>=4.41.0 \
    datasets \
    accelerate \
    evaluate \
    bitsandbytes \
    nltk \
    pandas \
    scipy

print("\nDone.")

In [ ]:
# S3: Patch compatibility issues in training scripts (adds **kwargs to compute_loss,
# replaces deprecated self.tokenizer with self.processing_class)
import os

SCRIPTS = "/content/concept_aware_training/transformers/examples/pytorch/language-modeling"

def patch_file(path, replacements):
    with open(path) as f:
        content = f.read()
    for old, new in replacements:
        content = content.replace(old, new)
    with open(path, 'w') as f:
        f.write(content)
    print(f"  Patched: {os.path.basename(path)}")

# Patch run_clm scripts: add **kwargs
for script in ['run_clm.py', 'run_clm_syn_custom_loss.py', 'run_clm_hyp_custom_loss.py']:
    patch_file(os.path.join(SCRIPTS, script), [
        ("def compute_loss(self, model, inputs, return_outputs=False):",
         "def compute_loss(self, model, inputs, return_outputs=False, **kwargs:")
    ])

# Patch custom_trainer.py: **kwargs + tokenizer → processing_class
patch_file(os.path.join(SCRIPTS, 'custom_trainer.py'), [
    ("def compute_loss(self, model, inputs, return_outputs=False):",
     "def compute_loss(self, model, inputs, return_outputs=False, **kwargs):"),
    ("self.tokenizer = tokenizer", "self.processing_class = tokenizer"),
    ("self.tokenizer(", "self.processing_class("),
    ("self.tokenizer.bos_token_id", "self.processing_class.bos_token_id"),
    ("self.tokenizer.eos_token_id", "self.processing_class.eos_token_id"),
])

print("All patches applied.")

In [ ]:
# S4: Mount Google Drive (for saving outputs)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# S5: Configure all paths
import os

# ── Where baseline checkpoints were saved (from previous training runs) ──
OUTPUT_ROOT = "/content/drive/MyDrive/concept_aware_outputs"  # UPDATE if different

# ── Repo data directory (data lives in the cloned repo, not Drive) ──
DATA_ROOT = "/content/concept_aware_training/data"

# ── Scripts directory ──
SCRIPTS = "/content/concept_aware_training/transformers/examples/pytorch/language-modeling"
REPO    = "/content/concept_aware_training"

# ── Data files ──
SYN_TRAIN = os.path.join(DATA_ROOT, "syn/youtube/context_loss_train.csv")
SYN_VAL   = os.path.join(DATA_ROOT, "syn/youtube/context_loss_val.csv")
HYP_TRAIN = os.path.join(DATA_ROOT, "hyp/youtube/dict_loss_train.csv")
HYP_VAL   = os.path.join(DATA_ROOT, "hyp/youtube/dict_loss_val.csv")

# Vanilla (plain text) val file — used for Task 1 NTP eval
# Check which one exists:
for candidate in [
    os.path.join(DATA_ROOT, "hyp/youtube/vanilla_val.txt"),
    os.path.join(DATA_ROOT, "syn/youtube/vanilla_val.txt"),
    os.path.join(DATA_ROOT, "hyp/youtube/context_syn_val.txt"),
]:
    if os.path.exists(candidate):
        VANILLA_VAL = candidate
        print(f"Vanilla val: {VANILLA_VAL}")
        break
else:
    print("WARNING: No vanilla val file found — update VANILLA_VAL manually below")
    VANILLA_VAL = os.path.join(DATA_ROOT, "hyp/youtube/vanilla_val.txt")  # set manually

# ── Baseline checkpoints (must exist from previous runs) ──
CKPT_VANILLA = os.path.join(OUTPUT_ROOT, "standard_syn")
CKPT_SYN_NCP = os.path.join(OUTPUT_ROOT, "syn_custom_loss")
CKPT_HYP_NCP = os.path.join(OUTPUT_ROOT, "hyp_custom_loss")

# Verify baselines exist
print("\nBaseline checkpoint status:")
for label, path in [("standard_syn", CKPT_VANILLA), ("syn_custom_loss", CKPT_SYN_NCP), ("hyp_custom_loss", CKPT_HYP_NCP)]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {label}: {path}")

In [ ]:
# S6: Download base model (Llama-3.2-1B)
# Skip if already downloaded
MODEL_LOCAL_PATH = "/content/Llama-3.2-1B"

if os.path.exists(MODEL_LOCAL_PATH) and os.path.exists(os.path.join(MODEL_LOCAL_PATH, 'config.json')):
    print(f"Model already at {MODEL_LOCAL_PATH} — skipping download")
else:
    from huggingface_hub import login, snapshot_download
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')  # store your token in Colab Secrets
    except Exception:
        hf_token = input("Paste your Hugging Face token: ").strip()
    login(token=hf_token)
    snapshot_download(
        repo_id="meta-llama/Llama-3.2-1B",
        local_dir=MODEL_LOCAL_PATH,
        ignore_patterns=["*.pt", "original/*"],
    )
    print(f"Downloaded to {MODEL_LOCAL_PATH}")

---
## Task 1 — NTP Baseline Evaluation

**What:** Evaluate all three existing checkpoints (standard CLM, synonym NCP, hypernym NCP) using **standard next-token prediction loss on vanilla (plain text) validation data** — no concept augmentation in the eval.

**Why:** Chen asked whether the previously reported numbers (eval_loss 3.90 / 1.06 / 0.79) were on vanilla or augmented data. They were on augmented data. This gives the corrected apples-to-apples comparison.

**Expected output:** A table of eval_loss, perplexity, and accuracy per checkpoint, saved to `task1_ntp_results.json`.

In [ ]:
import os, json
import pandas as pd

TASK1_RESULTS = '/content/task1_ntp_results.json'

# Only pass checkpoints that actually exist
baselines = [
    p for p in [CKPT_VANILLA, CKPT_SYN_NCP, CKPT_HYP_NCP]
    if os.path.exists(p)
]
if not baselines:
    print("ERROR: No baseline checkpoints found. Check OUTPUT_ROOT in Setup.")
else:
    ckpt_args = ' '.join(baselines)
    print(f"Evaluating {len(baselines)} checkpoints against: {VANILLA_VAL}")
    print()
    %cd {SCRIPTS}
    !python3 eval_ntp_baselines.py \
        --checkpoints {ckpt_args} \
        --validation_file {VANILLA_VAL} \
        --results_json {TASK1_RESULTS}

    results = json.load(open(TASK1_RESULTS))
    df1 = pd.DataFrame(results)[['checkpoint', 'eval_loss', 'perplexity', 'accuracy']]
    df1['checkpoint'] = df1['checkpoint'].apply(os.path.basename)
    print("\n=== Task 1 Results (vanilla NTP eval) ===")
    print(df1.sort_values('eval_loss').to_string(index=False))

---
## Task 2 — Differentiable NCP Training

**What:** Train a new model variant using a fully differentiable NCP loss. The original `custom_trainer.py` computes concept scores under `torch.no_grad()`, which kills gradient flow through the concept term. This version uses `torch.logsumexp` over the existing forward pass logits — zero extra compute, gradients flow fully.

**Loss:** `L = L_CLM + alpha * mean(-log Σ p(c | context))` for c in concept set C.

**Run syn first. Run hyp after if time permits.**

In [ ]:
# Task 2a: Differentiable NCP — Synonym data
CKPT_DIFF_NCP_SYN = os.path.join(OUTPUT_ROOT, 'diff_ncp_syn')

%cd {SCRIPTS}
!python3 run_clm_differentiable_ncp.py \
    --model_name_or_path {MODEL_LOCAL_PATH} \
    --train_file {SYN_TRAIN} \
    --validation_file {SYN_VAL} \
    --ncp_alpha 1.0 \
    --block_size 128 \
    --torch_dtype bfloat16 \
    --bf16 True \
    --gradient_accumulation_steps 8 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --auto_find_batch_size True \
    --overwrite_output_dir \
    --do_train \
    --do_eval \
    --output_dir {CKPT_DIFF_NCP_SYN}

In [ ]:
# Task 2b: Differentiable NCP — Hypernym data (run after 2a)
CKPT_DIFF_NCP_HYP = os.path.join(OUTPUT_ROOT, 'diff_ncp_hyp')

%cd {SCRIPTS}
!python3 run_clm_differentiable_ncp.py \
    --model_name_or_path {MODEL_LOCAL_PATH} \
    --train_file {HYP_TRAIN} \
    --validation_file {HYP_VAL} \
    --ncp_alpha 1.0 \
    --block_size 128 \
    --torch_dtype bfloat16 \
    --bf16 True \
    --gradient_accumulation_steps 8 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --auto_find_batch_size True \
    --overwrite_output_dir \
    --do_train \
    --do_eval \
    --output_dir {CKPT_DIFF_NCP_HYP}

---
## Task 3 — Build Contrastive Dataset with Hard Negatives

**What:** For each context prefix and positive concept set in the YouTube data, mine **hard negatives** from WordNet using three strategies (in priority order):
1. Co-hyponyms — siblings in the WordNet hierarchy
2. Wrong-sense distractors — other synsets of the same word (Chen's preference from email)
3. Same-POS fallback — random same-POS WordNet words

**Output:** `data/contrastive/youtube/contrastive_train.csv` with columns `text`, `positives`, `negatives`.

In [ ]:
# Task 3a: Install NLTK and download WordNet
!pip install nltk -q
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
print("WordNet ready.")

In [ ]:
# Task 3b: Build contrastive dataset
CONTRASTIVE_DIR = os.path.join(DATA_ROOT, 'contrastive/youtube')

%cd {REPO}
!python3 build_contrastive_dataset.py \
    --source both \
    --max_negatives 10 \
    --output_dir {CONTRASTIVE_DIR}

CONTRASTIVE_TRAIN = os.path.join(CONTRASTIVE_DIR, 'contrastive_train.csv')
CONTRASTIVE_VAL   = os.path.join(CONTRASTIVE_DIR, 'contrastive_val.csv')
print(f"\nTrain: {CONTRASTIVE_TRAIN}")
print(f"Val:   {CONTRASTIVE_VAL}")

In [ ]:
# Task 3c: Inspect contrastive dataset — verify negatives look reasonable
import pandas as pd, ast

df_c = pd.read_csv(CONTRASTIVE_TRAIN)
print(f"Rows: {len(df_c)}  |  Columns: {df_c.columns.tolist()}")
print()

# Check WordNet coverage: how many rows have at least 1 negative
has_neg = df_c['negatives'].apply(lambda x: len(ast.literal_eval(str(x))) > 0)
print(f"WordNet coverage: {has_neg.sum()}/{len(df_c)} rows have ≥1 hard negative ({has_neg.mean()*100:.1f}%)")
print()

# Show 3 sample rows
for _, row in df_c.sample(3, random_state=42).iterrows():
    print("TEXT:     ", str(row['text'])[:80])
    print("POSITIVES:", row['positives'])
    print("NEGATIVES:", row['negatives'])
    print()

---
## Task 4 — Contrastive Training + Final Comparison

**What:** Train with an InfoNCE-style contrastive loss that uses both the positive concept set (from Task 3) and the hard negatives. All computed from the single existing forward pass — zero extra compute.

**Loss:**
```
L = L_CLM
  + alpha * mean( -log Σ p(c+ | context) )          # differentiable NCP on positives
  + beta  * mean( log(p_pos + p_neg) - log_p_pos )   # InfoNCE contrastive term
```

**Requires:** Task 3 output (`contrastive_train.csv`). Run Task 3 first.

In [ ]:
# Task 4a: Contrastive training
CKPT_CONTRASTIVE = os.path.join(OUTPUT_ROOT, 'contrastive')

%cd {SCRIPTS}
!python3 run_clm_contrastive.py \
    --model_name_or_path {MODEL_LOCAL_PATH} \
    --train_file {CONTRASTIVE_TRAIN} \
    --validation_file {CONTRASTIVE_VAL} \
    --ncp_alpha 0.5 \
    --contrast_beta 1.0 \
    --block_size 128 \
    --torch_dtype bfloat16 \
    --bf16 True \
    --gradient_accumulation_steps 8 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --auto_find_batch_size True \
    --overwrite_output_dir \
    --do_train \
    --do_eval \
    --output_dir {CKPT_CONTRASTIVE}

In [ ]:
# Task 4b: Final comparison — evaluate ALL models on vanilla NTP
# Run this after Tasks 1-4 are complete.
import os, json
import pandas as pd

FINAL_RESULTS = '/content/final_comparison_results.json'

# Include all checkpoints; skip any that don't exist yet
all_checkpoints = {
    'standard_syn':   CKPT_VANILLA,
    'syn_ncp':        CKPT_SYN_NCP,
    'hyp_ncp':        CKPT_HYP_NCP,
    'diff_ncp_syn':   CKPT_DIFF_NCP_SYN,
    'diff_ncp_hyp':   CKPT_DIFF_NCP_HYP,
    'contrastive':    CKPT_CONTRASTIVE,
}
existing = {k: v for k, v in all_checkpoints.items() if os.path.exists(v)}
print(f"Found {len(existing)}/{len(all_checkpoints)} checkpoints:")
for k, v in existing.items():
    print(f"  {k}: {v}")

ckpt_args = ' '.join(existing.values())
%cd {SCRIPTS}
!python3 eval_ntp_baselines.py \
    --checkpoints {ckpt_args} \
    --validation_file {VANILLA_VAL} \
    --results_json {FINAL_RESULTS}

# Display final results table
results = json.load(open(FINAL_RESULTS))
df_final = pd.DataFrame(results)[['checkpoint', 'eval_loss', 'perplexity', 'accuracy']]
df_final['checkpoint'] = df_final['checkpoint'].apply(os.path.basename)
df_final = df_final.sort_values('eval_loss').reset_index(drop=True)
print("\n=== Final Results (all models, vanilla NTP eval) ===")
print(df_final.to_string(index=False))